Beyond RDS, Aurora, DynamoDB, and ElastiCache, AWS offers a suite of purpose-built databases — each optimised for a specific data model or access pattern. This notebook covers Redshift for data warehousing, and the specialised database services: DocumentDB, Neptune, Keyspaces, QLDB, Timestream, and MemoryDB. The SAA-C03 exam tests your ability to select the right database for a given scenario.

## Amazon Redshift

Redshift is AWS's fully managed **data warehouse** — optimised for **OLAP** (Online Analytical Processing): complex queries over large datasets, aggregations, and business intelligence workloads.

### OLTP vs OLAP

| | OLTP (RDS/Aurora) | OLAP (Redshift) |
|---|---|---|
| Workload | Many small transactions (INSERT/UPDATE/SELECT) | Few complex queries over huge datasets |
| Query type | Row-level, fast point lookups | Full scans, aggregations (SUM, AVG, GROUP BY) |
| Data size | GB to low TB | TB to PB |
| Users | Application | Analysts, BI tools |
| Storage | Row-oriented | **Columnar** |

### Why columnar storage?
In a row store, a query like `SELECT SUM(revenue) FROM orders` reads every column of every row. In a **columnar store**, only the `revenue` column is read. For analytical queries that touch a few columns across millions of rows, columnar storage reduces I/O by 10–100×.

## Redshift Architecture

### Cluster
A Redshift cluster has:
- **Leader node**: receives queries, parses them, builds execution plans, coordinates compute nodes, returns results to clients
- **Compute nodes**: execute query fragments in parallel, store data on local disks

```text
BI Tool / SQL Client
        │
        ▼
    Leader Node  (query planning + coordination)
    /     |     \
Compute  Compute  Compute   (parallel execution + local storage)
Node 1   Node 2   Node 3
```

Redshift uses **MPP** (Massively Parallel Processing) — the leader distributes query work across compute nodes and combines results.

### Node types

| Type | Storage | Use case |
|---|---|---|
| **RA3** | Managed storage (S3-backed, scales independently of compute) | Recommended default; decouple storage and compute |
| **DC2** | Local SSD (fixed to node) | Compute-intensive with frequent access to all data; smaller datasets |

**RA3** is the recommended choice for most new workloads — you scale compute and storage independently, and pay only for what you use.

### Redshift Serverless
No cluster to manage — Redshift automatically scales compute capacity based on workload. You pay per RPU-hour (Redshift Processing Unit). Ideal for intermittent analytics, dev/test, and workloads with unpredictable demand.

### Multi-AZ
RA3 clusters support **Multi-AZ deployment** — the cluster spans two AZs with automatic failover. Data is stored in S3 (managed storage), so failover does not require data replication.

## Loading Data into Redshift

### COPY command
The most efficient way to load data is the `COPY` command — it loads data in parallel from S3, DynamoDB, EMR, or SSH hosts directly into compute nodes.

```sql
COPY orders
FROM 's3://my-bucket/data/orders/'
IAM_ROLE 'arn:aws:iam::123456789012:role/RedshiftS3Role'
FORMAT AS PARQUET;
```

- Reads multiple files in parallel (one per compute node slice)
- Supports CSV, JSON, Parquet, ORC, Avro
- Far faster than INSERT statements for bulk loads

### Redshift Spectrum
Redshift Spectrum lets you **query data directly in S3** without loading it into Redshift first.

```text
Redshift cluster → Spectrum layer → S3 data lake
                        ↑
        thousands of Spectrum nodes process S3 in parallel
```

- Define external tables in the **AWS Glue Data Catalog**
- Query them with standard SQL alongside Redshift tables
- Pay per TB scanned in S3
- Ideal for: cold data in S3, ad-hoc queries on raw data, data lake + warehouse combined queries

### ETL pipeline pattern
```text
Source DBs/Apps → Kinesis/Glue ETL → S3 (data lake) → COPY → Redshift
                                          │
                                   Spectrum queries ──────────────┘
```

## Redshift Key Features

### Snapshots and backup
- **Automated snapshots**: every 8 hours or every 5 GB of data changes; retention 1–35 days
- **Manual snapshots**: persist until deleted
- Snapshots stored in S3 — can be copied to another region for DR
- Restore creates a new cluster from snapshot

### Enhanced VPC Routing
Forces all COPY and UNLOAD traffic between Redshift and S3 to go through your VPC (and VPC endpoints) rather than the public internet. Required for compliance scenarios where data must not traverse the public internet.

### Distribution styles
How Redshift distributes rows across compute nodes affects query performance:

| Style | How | Use case |
|---|---|---|
| **AUTO** | Redshift chooses | Default |
| **EVEN** | Round-robin across nodes | No clear distribution key |
| **KEY** | By column value (same values on same node) | Joining large tables on the same column |
| **ALL** | Full copy on every node | Small dimension tables joined frequently |

### Sort keys
Define the order data is stored on disk within each node. Queries with `WHERE` or `ORDER BY` on the sort key skip large blocks of data (zone maps), dramatically reducing I/O.

## Amazon DocumentDB

DocumentDB is AWS's managed **document database**, compatible with **MongoDB** drivers and tools.

### Key properties
- Stores **JSON documents**
- MongoDB-compatible API (MongoDB 3.6, 4.0, 5.0)
- Same storage architecture as Aurora: data replicated 6 ways across 3 AZs; self-healing; auto-grows to 128 TB
- Up to 15 read replicas
- Continuous backup with PITR (35 days)

### Use cases
- Migrating existing MongoDB workloads to AWS without rewriting application code
- Content management, catalogs, user profiles — data with variable or nested structure
- Applications that use the MongoDB query language (MQL)

> **Exam tip:** "MongoDB-compatible" or "document database" → DocumentDB.

## Amazon Neptune

Neptune is AWS's fully managed **graph database**.

### What is a graph database?
Graph databases model data as **nodes** (entities) and **edges** (relationships), optimised for traversing complex, highly connected data.

```text
Alice ──[FOLLOWS]──▶ Bob ──[FOLLOWS]──▶ Carol
  │                   │
[LIKES]           [PURCHASED]
  │                   │
Post-A            Product-X
```

### Key properties
- Supports **Gremlin** (Apache TinkerPop) and **SPARQL** (W3C RDF) query languages
- Also supports **openCypher** (Cypher query language)
- Same Aurora-style storage: 6 copies across 3 AZs, up to 64 TB, self-healing
- Up to 15 read replicas, sub-10ms query latency
- Neptune Streams: change capture for graph changes

### Use cases
- **Social networks** — friend-of-friend queries, mutual connections
- **Fraud detection** — link analysis between accounts, devices, IPs
- **Knowledge graphs** — interconnected facts and relationships
- **Recommendation engines** — "users who bought X also bought Y"
- **Network and IT topology** — mapping infrastructure dependencies

> **Exam tip:** "Graph database", "social network", "fraud detection", "highly connected data" → Neptune.

## Amazon Keyspaces (for Apache Cassandra)

Keyspaces is a serverless, managed **Apache Cassandra-compatible** database.

### Key properties
- Compatible with **CQL** (Cassandra Query Language) and Cassandra drivers
- Serverless — automatically scales tables up/down, pay per request or provisioned throughput
- Single-digit millisecond latency at any scale
- Data replicated 3× across multiple AZs
- PITR up to 35 days

### Use cases
- Migrating existing Apache Cassandra workloads to AWS without managing clusters
- Industrial equipment data, IoT telemetry, time-series-like data at massive scale
- High-throughput write workloads with wide rows

> **Exam tip:** "Cassandra-compatible" or "migrate Cassandra" → Keyspaces.

## Amazon QLDB (Quantum Ledger Database)

QLDB is a fully managed **ledger database** with a cryptographically verifiable, **immutable transaction log**.

### Key properties
- Every change to data is recorded in an **append-only journal** — records can never be altered or deleted
- Each transaction is cryptographically hashed — you can mathematically prove no tampering has occurred
- Uses **PartiQL** (SQL-compatible) as the query language
- 2–3× better performance than blockchain frameworks for ledger use cases
- **Serverless** — no infrastructure to manage
- **Not decentralised** — AWS manages it (unlike true blockchain); no multi-party trust needed

### Use cases
- Financial transaction history — prove no record was altered
- Supply chain provenance — track every step of a product's journey
- Healthcare records — immutable audit trail
- Any scenario requiring a trusted, tamper-evident audit log

> **Exam tip:** "Immutable audit trail", "cryptographically verifiable history", "financial ledger" → QLDB. If the scenario requires **decentralised** multi-party trust → Managed Blockchain.

## Amazon Timestream

Timestream is a fully managed **time-series database** — optimised for storing and querying data points indexed by time.

### Key properties
- Automatically tiers data: recent data in memory → historical data on magnetic storage
- Built-in **time-series analytics functions** (smoothing, interpolation, approximation)
- **1,000× faster** and **1/10th the cost** of relational DBs for time-series workloads
- Serverless — scales automatically
- SQL-compatible query language with time-series extensions

### Use cases
- **IoT sensor data** — temperature, pressure, GPS coordinates over time
- **Application metrics** — CPU, memory, request latency over time
- **DevOps monitoring** — infrastructure telemetry
- **Financial tick data** — stock prices, trading volumes
- Any dataset where the primary dimension is time

> **Exam tip:** "IoT", "time-series", "sensor data", "metrics over time" → Timestream.

## Amazon MemoryDB for Redis

MemoryDB is a **durable, Redis-compatible** in-memory database — not just a cache.

### MemoryDB vs ElastiCache for Redis

| | ElastiCache Redis | MemoryDB for Redis |
|---|---|---|
| Primary role | Cache (in front of a DB) | **Primary database** (no separate DB needed) |
| Durability | Optional (RDB/AOF) | **Always durable** — Multi-AZ transaction log |
| Data loss on failure | Possible | **None** — every write confirmed to transaction log |
| Latency reads | Microseconds | Microseconds |
| Latency writes | Microseconds | Single-digit milliseconds (log commit) |
| Use case | Acceleration layer | Microservices needing fast + durable primary store |

### Use cases
- Microservices that need microsecond reads, low-latency writes, and full durability without a separate database
- Session management where data must survive node failure
- Gaming leaderboards, real-time bidding — needs Redis API + guaranteed durability

> **Exam tip:** "Durable Redis", "Redis as primary database", "can't afford to lose cache data" → MemoryDB. Pure caching → ElastiCache.

## Database Selection Guide

| Scenario | Service |
|---|---|
| Relational, MySQL/PostgreSQL, high performance | Aurora |
| Relational, Oracle / SQL Server | RDS |
| NoSQL key-value / document, serverless scale | DynamoDB |
| In-memory cache, microsecond reads | ElastiCache (Redis) |
| Durable Redis, primary database | MemoryDB |
| Data warehouse, OLAP, BI analytics, PB scale | Redshift |
| Query S3 data lake with SQL | Redshift Spectrum / Athena |
| MongoDB-compatible documents | DocumentDB |
| Graph — social network, fraud, recommendations | Neptune |
| Cassandra-compatible, managed | Keyspaces |
| Immutable audit trail, financial ledger | QLDB |
| Decentralised multi-party ledger | Managed Blockchain |
| Time-series, IoT, metrics, sensor data | Timestream |

## Working with Redshift and Other Databases via boto3

In [ ]:
import boto3

redshift    = boto3.client('redshift',    region_name='us-east-1')
docdb       = boto3.client('docdb',       region_name='us-east-1')
timestream  = boto3.client('timestream-write', region_name='us-east-1')

In [ ]:
# Create a Redshift RA3 cluster
redshift.create_cluster(
    ClusterIdentifier='prod-warehouse',
    NodeType='ra3.xlplus',
    NumberOfNodes=2,               # 2 compute nodes (+ 1 leader)
    MasterUsername='admin',
    MasterUserPassword='Change-Me-123!',
    DBName='analytics',
    ClusterSubnetGroupName='redshift-subnet-group',
    VpcSecurityGroupIds=['sg-0redshift'],
    Encrypted=True,
    EnhancedVpcRouting=True,       # COPY/UNLOAD traffic stays in VPC
    AutomatedSnapshotRetentionPeriod=7,
    Tags=[{'Key': 'Environment', 'Value': 'production'}]
)
print("Redshift cluster creation initiated")

In [ ]:
# Run SQL on Redshift using the Data API (no persistent connection needed)
data_api = boto3.client('redshift-data', region_name='us-east-1')

# Execute a query asynchronously
response = data_api.execute_statement(
    ClusterIdentifier='prod-warehouse',
    Database='analytics',
    SecretArn='arn:aws:secretsmanager:us-east-1:123456789012:secret:redshift-creds',
    Sql="""
        SELECT date_trunc('month', order_date) AS month,
               SUM(revenue)                    AS total_revenue,
               COUNT(DISTINCT customer_id)     AS unique_customers
        FROM   fact_orders
        WHERE  order_date >= '2026-01-01'
        GROUP  BY 1
        ORDER  BY 1
    """
)
statement_id = response['Id']
print(f"Query submitted: {statement_id}")

In [ ]:
# Write IoT sensor data to Timestream
import time

timestream.write_records(
    DatabaseName='IoTData',
    TableName='SensorReadings',
    Records=[
        {
            'Dimensions': [
                {'Name': 'device_id',  'Value': 'sensor-001'},
                {'Name': 'location',   'Value': 'warehouse-a'},
            ],
            'MeasureName':  'temperature',
            'MeasureValue': '23.5',
            'MeasureValueType': 'DOUBLE',
            'Time': str(int(time.time() * 1000)),
            'TimeUnit': 'MILLISECONDS'
        },
        {
            'Dimensions': [
                {'Name': 'device_id',  'Value': 'sensor-001'},
                {'Name': 'location',   'Value': 'warehouse-a'},
            ],
            'MeasureName':  'humidity',
            'MeasureValue': '65.2',
            'MeasureValueType': 'DOUBLE',
            'Time': str(int(time.time() * 1000)),
            'TimeUnit': 'MILLISECONDS'
        }
    ]
)
print("Sensor readings written to Timestream")

In [ ]:
# Neptune — graph traversal using Gremlin (via HTTP/WebSocket)
# boto3 manages the cluster; queries go via the Gremlin endpoint

# Create a Neptune cluster
neptune = boto3.client('neptune', region_name='us-east-1')
neptune.create_db_cluster(
    DBClusterIdentifier='social-graph',
    Engine='neptune',
    DBSubnetGroupName='neptune-subnet-group',
    VpcSecurityGroupIds=['sg-0neptune'],
    StorageEncrypted=True,
    BackupRetentionPeriod=7,
    EnableIAMDatabaseAuthentication=True
)
print("Neptune cluster creation initiated")

# Gremlin query example (executed via HTTP against Neptune endpoint):
# g.V().has('person', 'name', 'Alice')
#  .out('FOLLOWS')            # Alice's direct follows
#  .out('FOLLOWS')            # friend-of-friend
#  .values('name').dedup()    # unique names

## Summary

| Service | Type | Key Takeaway |
|---|---|---|
| Redshift | Columnar data warehouse | OLAP; MPP; COPY from S3; Spectrum queries S3 directly |
| Redshift RA3 | Managed storage nodes | Decouple compute and storage; Multi-AZ supported |
| Redshift Serverless | Serverless warehouse | Pay per RPU-hour; auto-scales; no cluster management |
| Redshift Spectrum | S3 query layer | SQL queries on S3 data lake without loading into Redshift |
| Enhanced VPC Routing | Network control | COPY/UNLOAD traffic stays in VPC |
| DocumentDB | Document (MongoDB) | MongoDB-compatible; same Aurora storage architecture |
| Neptune | Graph database | Nodes + edges; social networks, fraud, recommendations |
| Neptune query languages | Graph queries | Gremlin, SPARQL, openCypher |
| Keyspaces | Cassandra-compatible | Serverless; managed Cassandra; no cluster ops |
| QLDB | Immutable ledger | Append-only; cryptographically verifiable; PartiQL |
| QLDB vs Managed Blockchain | Centralised vs decentralised | QLDB = trusted central authority; Blockchain = multi-party trust |
| Timestream | Time-series | IoT, metrics, sensor data; auto-tiering memory → magnetic |
| MemoryDB | Durable Redis | Redis as primary DB; every write durable; no data loss |
| MemoryDB vs ElastiCache | Primary DB vs cache | MemoryDB = durable primary; ElastiCache = acceleration cache |